<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #2c3e50;
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    TEEHR for NextGen Model Output Analysis and Evaluation
</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Ayman Nassar <a href="mailto:ayman.nassar@usu.edu">(ayman.nassar@usu.edu)</a></li>
<li>David Tarboton <a href="mailto:david.tarboton@usu.edu">(david.tarboton@usu.edu)</a></li>
</ul>

**Last Updated:** 05/15/2026

**Purpose:**

**TEEHR** (**T**ools for **E**xploratory **E**valuation in **H**ydrologic **R**esearch) is a Python package that provides a flexible framework for evaluating hydrologic model performance. In this notebook, **TEEHR** is used to assess NextGen model outputs by comparing simulated streamflow with `USGS observations` and the `National Water Model Retrospective Analysis v3.0`.
The workflow demonstrates how to set up the TEEHR evaluation environment, organize required datasets, and generate performance metrics and diagnostic plots. More information about **TEEHR** can be found in the 
<span style="background-color:#e0f4ff; padding:4px 6px; border-radius:6px;">
  <a href="https://rtiinternational.github.io/teehr/user_guide/index.html#user-guide" target="_blank" style="font-weight:bold; color:#005a9e;">
    TEEHR User Guide
  </a>
</span>.

**Audience:**

Researchers, hydrologists, practitioners, and graduate students working with NextGen hydrologic simulations. Users should be familiar with Python, Jupyter Notebooks, and basic hydrologic modeling concepts.

**Description:**

<div style="font-size:14px; margin-bottom:0; padding-bottom:0;">

This notebook leverages **TEEHR** to load, organize, and evaluate hydrologic datasets needed to assess NextGen streamflow simulations.  
It performs the following tasks:
- Reads `NextGen-simulated streamflow` outputs.  
- Aligns them with `USGS observed discharge` and `NWM v3.0 retrospective simulations`.  
- Sets up or clones a `TEEHR evaluation directory`.  
- Computes performance metrics and generates diagnostic plots for model evaluation.
</div>
<div style="margin-top:-16px;"></div>

**Data Description:**

This notebook uses three hydrologic datasets organized according to **TEEHR’s** primary–secondary evaluation structure:

**1. USGS Observed Streamflow** (`Primary Dataset`): Ground-truth discharge time series from the U.S. Geological Survey (NWIS).  
This dataset serves as the **primary reference** against which all model outputs are evaluated.

**2. NextGen Model Outputs** (`Secondary Dataset`): Simulated streamflow generated by the **NextGen hydrologic model** for each evaluation location. Loaded in TEEHR as a **secondary dataset**.

**3. NWM Retrospective Analysis v3.0** (`Secondary Dataset`): Simulated streamflow from the `National Water Model v3.0 retrospective dataset`.  
Also treated as a **secondary dataset**.

Supporting metadata (gage identifiers, attributes, evaluation configuration files) ensures proper `spatial and temporal alignment` across datasets.  

**Software Requirements:**

This notebook uses the following library versions:

> shutil: 3.11.8  
> Path: 3.11.8  
> typing: 3.11.8  
> re: 2.2.1  
> logging: 0.5.1.2  
> pandas: 2.2.1  
> xarray: 2024.3.0  
> hvplot: 0.9.2  
> datetime: 3.11.8  
> bokeh: 3.8.1  
> teehr: 0.5.2  

It also uses helper functions from **`ngiab_utils.py`**.

<div style="
    padding: 15px 20px; 
    background-color: #e2f0fe; 
    border-left: 6px solid #3b82f6; 
    color: #1e3a8a; 
    border-radius: 4px; 
    margin-bottom: 20px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
">
    <h3 style="margin-top: 0; color: #1e3a8a; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        💡 Quick Note Before You Begin
    </h3>
    <p style="margin-bottom: 10px; font-size: 1.05em;">
        To make sure everything runs smoothly and all settings initialize correctly, <strong>please take a moment to restart the kernel before running the cells below.</strong>
    </p>
    <p style="margin: 0; font-size: 0.95em;">
        <strong>How to do this:</strong> Simply navigate to the <strong>Kernel</strong> menu at the top and select <span style="background-color: rgba(0,0,0,0.05); padding: 2px 6px; border-radius: 3px; border: 1px solid rgba(0,0,0,0.1);"><strong>“Restart Kernel and Clear Outputs of All Cells”</strong></span>. Thank you!
    </p>
</div>

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    1. Prepare the Python Environment
  </h3>
</div>

In this step, we prepare the **Python environment** by importing all required libraries and utilities used throughout this TEEHR evaluation workflow.  
These modules support **data processing**, **time-series visualization**, and **NextGen/TEEHR evaluation**.

In [ ]:
import shutil
from pathlib import Path
from typing import Union
import re
import logging
from dataretrieval import nwis
import geopandas as gpd
from shapely.geometry import Point
import requests

import pandas as pd
import xarray as xr
import hvplot.pandas
from datetime import datetime

from bokeh.io import output_notebook
output_notebook()

from teehr.evaluation.utils import print_tree
import teehr
import ngiab_utils

logger = logging.getLogger(__name__)

print(f"TEEHR version: {teehr.__version__}")

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    2. Set Inputs
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
In this step, you define the <strong>key inputs</strong> required for the TEEHR workflow.  
These inputs specify the <strong>hydrofabric domain</strong> you are working with and the <strong>directories used for preprocessing and data preparation</strong>.
</p>

<div style="background-color:#f3f6ff; border-left: 5px solid #6c63ff; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; color: #000000; margin-top: 10px;">
  <strong>Inputs Defined in the Next Cell:</strong>
  <ul style="margin-top: 8px; margin-bottom: 0;">
    <li><strong>Hydrofabric ID:</strong> Specifies the spatial domain used for this NGIAB run.  
      In this example, the domain is a USGS gage:
      <ul>
        <li><code>gage-10109001</code></li>
      </ul>
    </li>
    <li><strong>Preprocessing Directory:</strong>  
      The path to the NGIAB preprocessing output associated with the selected hydrofabric ID.
      <br><em>Example:</em>  
      <code>/home/jovyan/ngiab_preprocess_output/gage-10109001</code>
    </li>
    <li><strong>Temporary Output Directory:</strong>  
      This folder stores the TEEHR evaluation files, including metadata, time series, and supporting scripts.
      <br><em>Example:</em>  
      <code>/home/jovyan/ngiab_preprocess_output/gage-10109001/teehr</code>
    </li>
  </ul>
</div>

In [ ]:
# Hydrofabric/gage ID for this NGIAB run
hydrofabric_id = "gage-10109001"

# Path to the NGIAB preprocessing output for this gage
SAMPLE_NGIAB_OUTPUT_DIR = Path(f"/home/jovyan/ngiab_preprocess_output/{hydrofabric_id}")

# Directory containing all TEEHR evaluation data and supporting files.
TEMP_DIR = Path(f"/home/jovyan/ngiab_preprocess_output/{hydrofabric_id}/teehr")

# Create the temp directory if it doesn't exist
TEMP_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NGIAB_OUTPUT_DIR, TEMP_DIR

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    3. Get a Table Mapping NGIAB IDs to USGS Gage IDs
  </h3>
</div>

<p style="margin-top:0; padding-top:4px;">
This step extracts the <strong>USGS gage identifiers</strong> from the hydrofabric and generates a lookup table that links each 
<strong>NGIAB segment ID</strong> (<code>ngen</code>) to its corresponding <strong>USGS gage ID</strong>.  
This mapping ensures that discharge records from NGIAB outputs aligned with observed USGS streamflow data.
</p>

<div style="background-color:#f3f6ff; border-left: 5px solid #6c63ff; padding: 14px; border-radius: 8px; font-family: 'Segoe UI', sans-serif; font-size: 15px; color: #000000; margin-top: 10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top: 8px; margin-bottom: 0;">
    <li>The hydrofabric is queried to extract all gages associated with the selected domain.</li>
    <li>A correspondence table is created linking:
      <ul>
        <li><strong>NGIAB routing IDs</strong> (model segment identifiers)</li>
        <li><strong>USGS gage IDs</strong> (e.g., <code>10109001</code>)</li>
      </ul>
    </li>
      </ul>
    </li>
  </ul>
</div>

In [ ]:
# Extract gage information from the hydrofabric directory.
# This returns a list of tuples in the format: (ngen_segment_id, usgs_gage_id)
ngiab_usgs_gages = ngiab_utils.get_gages_from_hydrofabric(SAMPLE_NGIAB_OUTPUT_DIR)

# Convert the list of gages into a clean Pandas DataFrame
ngiab_gages_df = pd.DataFrame(ngiab_usgs_gages, columns=["ngen", "usgs"])

# Standardize USGS gage ID format by adding a "usgs-" prefix
# Example: 10109001  →  usgs-10109001
ngiab_gages_df["usgs"] = "usgs-" + ngiab_gages_df["usgs"].astype(str)

# Normalize NGIAB routing segment names by replacing "wb" with "ngen"
# Example: wb-123  →  ngen-123
ngiab_gages_df["ngen"] = ngiab_gages_df["ngen"].str.replace("wb", "ngen")

# Display the final crosswalk table
# This links each NGIAB segment (ngen) to its corresponding USGS gage ID
ngiab_gages_df

**Explanation of the output/result**

The DataFrame `ngiab_gages_df` shows the mapping between NGIAB model segment IDs and their corresponding USGS gage IDs.  
For example, in this case the row indicates that the NGIAB segment (e.g., `ngen-2861391`) is linked to a real-world USGS streamflow gage (e.g., `usgs-10109001`).  
These IDs may vary depending on the watershed or hydrofabric used, but the table always provides the correct NGIAB–USGS pairing for the dataset being analyzed.

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    4. Get the NGIAB Simulation Start and End Times
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
This step reads the NGIAB configuration files to extract the simulation’s 
<strong>start</strong> and <strong>end timestamps</strong>.  
These values define the valid time window for the model outputs and ensure that all subsequent 
processing steps operate within the correct temporal bounds.
</p>

<div style="background-color:#f3f6ff; border-left:5px solid #6c63ff; padding:14px; border-radius:8px;
            font-family:'Segoe UI', sans-serif; font-size:15px; color:#000000; margin-top:10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top:8px; margin-bottom:0;">
    <li>The workflow searches the NGIAB preprocessing directory for the model’s time configuration file.</li>
    <li>It extracts the defined <strong>simulation start</strong> and <strong>end times</strong> as Python datetime objects.</li>
  </ul>
</div>

In [ ]:
start_date, end_date = ngiab_utils.get_simulation_start_end_time(
    SAMPLE_NGIAB_OUTPUT_DIR
)
start_date, end_date

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    5. Create an Evaluation Object and Directory Using an Empty Template
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
In this step, an empty <strong>TEEHR Evaluation</strong> object is created along with a dedicated 
directory where all evaluation inputs, outputs, and configuration files will be stored.  
This provides a structured workspace for organizing the results.
</p>

<div style="background-color:#f3f6ff; border-left:5px solid #6c63ff; padding:14px; border-radius:8px;
            font-family:'Segoe UI', sans-serif; font-size:15px; color:#000000; margin-top:10px;">
  <strong>What Happens in This Step:</strong>
  <ul style="margin-top:8px; margin-bottom:0;">
    <li>A new TEEHR evaluation folder is created (if it does not already exist).</li>
    <li>An empty <code>Evaluation</code> object is initialized using the TEEHR template structure.</li>
    <li>This directory will later store model outputs, observed data, alignment tables, 
        and evaluation metrics generated during the workflow.</li>
  </ul>
</div>


In [ ]:
# create evaluation object

teehr_dir = Path(SAMPLE_NGIAB_OUTPUT_DIR, "teehr")
if teehr_dir.exists():
    shutil.rmtree(teehr_dir)
ev = teehr.Evaluation(dir_path=teehr_dir, create_dir=True)
ev.clone_template()

In [ ]:
# create df summarizing s3 evals
s3_df = ev.list_s3_evaluations()
s3_df.head()

In [ ]:
# get bucket url for e3
e3_row = s3_df[s3_df['name'] == 'e3_usgs_hourly_streamflow']
e3_url = e3_row['url'].values[0]

# format the url
e3_url = e3_url.replace("s3://", "s3a://")
s3_dataset_url = f"{e3_url}/dataset"
s3_crosswalk_url = f"{s3_dataset_url}/location_crosswalks/"

print(s3_crosswalk_url)

In [ ]:
# query the location_crosswalk table from the s3 bucket ahead of cloning
s3_crosswalk_sdf = ev.spark.read.parquet(s3_crosswalk_url)
s3_crosswalk_sdf

In [ ]:
# convert to pandas for ease of use with your 'ngiab_gages_df'
s3_crosswalk_df = s3_crosswalk_sdf.toPandas()
s3_crosswalk_df.head(20)

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Verify USGS Gage IDs in the Crosswalk Table
</h4>

<p style="font-family:'Segoe UI', sans-serif; line-height:1.55; margin-top:10px; color:#2b2b2b;">
In this step, we verify that the USGS gage ID exist in the crosswalk table.
</p>

In [ ]:
usgs_gage = "usgs-10109001"

row = s3_crosswalk_df[s3_crosswalk_df["primary_location_id"] == usgs_gage]

print(row if not row.empty else "USGS gage not found")

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    6. Register USGS Gage Metadata and Define Location Crosswalk
  </h3>
</div>

<p style="margin-top:0; padding-top:6px;">
The USGS gage used in this case study is not available in the existing TEEHR dataset bucket.
Therefore, we first retrieve the gage metadata (location and name) directly from the USGS NWIS service
and register it as a location in the TEEHR evaluation environment. This step enables subsequent
streamflow data retrieval and allows us to define a location crosswalk between the USGS gage and the
corresponding NextGen model segment.
</p>


In [ ]:
# Register USGS gage location + create USGS↔NGEN crosswalk in TEEHR ---

# Inputs
site_no = "10109001"
ngen_id = "ngen-2861391"

# 1) Fetch USGS site metadata (lat/lon + station name) from NWIS
site = nwis.get_record(sites=site_no, service="site").iloc[0]
pt = Point(float(site["dec_long_va"]), float(site["dec_lat_va"]))

# 2) Register locations in TEEHR (USGS gage + matching NGEN segment)
loc_gdf = gpd.GeoDataFrame(
    [
        {"id": f"usgs-{site_no}", "name": str(site["station_nm"]), "geometry": pt},
        {"id": ngen_id,           "name": f"NGEN {ngen_id.split('-')[-1]}", "geometry": pt},  # same point is OK for now
    ],
    crs="EPSG:4326",
)
ev.locations.load_dataframe(loc_gdf)

# 3) Define the location crosswalk (USGS = primary, NGEN = secondary)
ev.location_crosswalks.load_dataframe(
    pd.DataFrame([{"primary_location_id": f"usgs-{site_no}", "secondary_location_id": ngen_id}])
)

print(f"Registered location: usgs-{site_no} ({site['station_nm']}) and crosswalk → {ngen_id}")

In [ ]:
# Verify that the USGS gage has been registered in the location crosswalk table
ev.location_crosswalks.to_pandas()

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    7. Fetching USGS Streamgage Data
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
In this step, we use the TEEHR’s built-in tools to fetch USGS streamflow data
</p>

In [ ]:
# Fetch USGS streamflow observations for all registered USGS gages in the TEEHR
ev.fetch.usgs_streamflow(
    start_date=start_date,
    end_date=end_date
)

In [ ]:
# Check registered TEEHR configurations
ev.configurations.to_pandas()

In [ ]:
# Convert primary timeseries table to a Pandas DataFrame
usgs_df = ev.primary_timeseries.to_pandas()

# Plot the time series using TEEHR's Pandas accessor
usgs_df.teehr.timeseries_plot()

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    8. Fetch NWM Streamflow Data
  </h3>
</div>
<p style="margin-top:0; padding-top:6px;">
In this step, we use TEEHR’s built-in fetching utilities to retrieve National Water Model (NWM) streamflow data. 
The data are fetched automatically for all <b>secondary location IDs</b> listed in the location crosswalk table 
that are prefixed with the selected NWM version (e.g., <code>nwm20</code>, <code>nwm21</code>, <code>nwm30</code>). 
This mechanism enables seamless alignment between observed (USGS) and modeled (NWM) streamflow at corresponding locations.
For additional details, see the TEEHR documentation:
<a href="https://rtiinternational.github.io/teehr/api/generated/teehr.evaluation.fetch.Fetch.nwm_retrospective_points.html" target="_blank">
NWM Retrospective Points Fetch
</a>.
</p>


<h4 style="background-color:#e6ebff; color:#000000; padding:8px 12px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Retrieve NWM feature_id (NHDPlus COMID) for a USGS gage
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; color:#000000; margin-top:6px;">
In this step, we identify the <strong>NWM feature_id</strong> corresponding to a given <strong>USGS stream gage</strong>.  
The National Water Model (NWM) retrospective datasets are indexed by <strong>NHDPlus COMID</strong>,  
not by USGS gage IDs. Therefore, we use the <strong>USGS NLDI service</strong> to translate a USGS gage ID  
into its associated COMID, which can then be used to fetch NWM streamflow data.
</p>


In [ ]:
# USGS gage ID
gage_id = "10109001"

# Query USGS NLDI for the gage → NHDPlus COMID
url = f"https://api.water.usgs.gov/nldi/linked-data/nwissite/USGS-{gage_id}"
response = requests.get(url, timeout=30)
response.raise_for_status()

data = response.json()

# Basic validation
features = data.get("features", [])
if not features:
    raise ValueError(f"No NLDI feature found for USGS-{gage_id}")

nwm_feature_id = features[0].get("properties", {}).get("comid")
if not nwm_feature_id:
    raise ValueError(f"NHDPlus COMID not found for USGS-{gage_id}")

nwm_feature_id = str(nwm_feature_id)

print(f"NWM feature_id (COMID) for USGS {gage_id}: {nwm_feature_id}")

<h4 style="background-color:#e6ebff; color:#000000; padding:8px 12px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Register NWM feature_id in the Location Crosswalk Table
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; color:#000000; margin-top:6px;">
In this step, we link the <strong>USGS gage location</strong> to its corresponding  
<strong>National Water Model (NWM) feature</strong> by adding an entry to the  
<strong>TEEHR location crosswalk table</strong>.  
This crosswalk explicitly maps the USGS gage ID to the NWM retrospective  
<strong>feature_id (NHDPlus COMID)</strong>, enabling direct comparison between  
observed USGS streamflow and simulated NWM outputs within the TEEHR framework.
</p>


In [ ]:
nwm_feature_id = "664424"  # e.g., "12345678"

nwm_cw = pd.DataFrame([{
    "primary_location_id": "usgs-10109001",
    "secondary_location_id": f"nwm30-{nwm_feature_id}",
}])

ev.location_crosswalks.load_dataframe(nwm_cw)

<h4 style="background-color:#e6ebff; color:#000000; padding:8px 12px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Fetch NWM Retrospective Streamflow Data
</h4>

<p style="font-family:'Segoe UI',sans-serif; font-size:15px; color:#000000; margin-top:6px;">
In this step, we use <strong>TEEHR’s built-in fetch utility</strong> to retrieve  
<strong>National Water Model (NWM) retrospective streamflow</strong> data for the  
specified simulation period.  

The fetch operation automatically uses the <strong> location crosswalk table to identify all secondary_location_id values prefixed with nwm30 and downloads the corresponding NWM streamflow time series.The fetched data are then registered as a primary dataset within  
the TEEHR evaluation environment, making them directly comparable to observed USGS streamflow.
</p>

In [ ]:
ev.fetch.nwm_retrospective_points(
    nwm_version="nwm30",
    variable_name="streamflow",
    start_date=start_date,
    end_date=end_date
)

<h4 style="background-color:#e6ebff; color:#000000; padding:8px 12px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:16px;">
🔹 Check the Crosswalk Table after Fetching the NWM Retrospective Streamflow Data
</h4>

In [ ]:
location_crosswalks_df = ev.location_crosswalks.to_pandas()
location_crosswalks_df

In [ ]:
usgs_df["value_time"] = pd.to_datetime(usgs_df["value_time"])
usgs_df = usgs_df.sort_values("value_time").reset_index(drop=True)
usgs_df

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:12px 16px; border-radius:8px; margin-top:0; margin-bottom:0;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    9. Retrieving Simulated Streamflow from NGIAB
  </h3>
</div>

<p style="margin-top:0; padding-top:6px;">
In this step, we retrieve simulated streamflow directly from the NGIAB <code>t-route</code> NetCDF output file. 
The extracted discharge time series are then prepared for subsequent evaluation and analysis workflows.
</p>

In [ ]:
#  Locate the NGIAB troute NetCDF output file
troute_output_file_list = list(
    Path(SAMPLE_NGIAB_OUTPUT_DIR, "outputs/troute").glob("*.nc")
)

# Select the first troute output file
troute_output_nc_filepath = troute_output_file_list[0]

# Open the troute routing dataset
troute_ds = xr.open_dataset(troute_output_nc_filepath)
print(troute_ds)

# Path where the temporary subset file will be saved
troute_subset_filepath = Path(TEMP_DIR, "troute_output_subset.nc")
print(troute_subset_filepath)

# Retrieve the validated NGIAB–USGS crosswalk from the Evaluation object
crosswalk_df = ev.location_crosswalks.to_pandas()
print(crosswalk_df)

# Extract the numeric NGIAB routing IDs (strip off the 'ngen-' prefix)
ngen_gages = [
    int(gage.split("-")[1])
    for gage in crosswalk_df.secondary_location_id.tolist()
    if gage.split("-")[0] == "ngen"
]

# Subset the troute dataset to the NGIAB routing segments of interest
troute_ds.sel(feature_id=ngen_gages).to_netcdf(troute_subset_filepath)

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Add NextGen Configuration to the TEEHR Evaluation Environment
</h4>

In [ ]:
NGIAB_CONFIGURATION_NAME = "ngen_simulation"

# Get current configurations as a DataFrame
configs_df = ev.configurations.to_pandas()

# Check if the NGIAB configuration already exists
if NGIAB_CONFIGURATION_NAME not in configs_df["name"].values:
    ev.configurations.add([
        teehr.Configuration(
            name=NGIAB_CONFIGURATION_NAME,
            type="secondary",
            description="Nextgen simulation output"
        )
    ])
else:
    print(f"Configuration '{NGIAB_CONFIGURATION_NAME}' already exists — skipping add.")

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff; border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:18px;">
  🔹 Review the Current Configurations
</h4>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.55; margin-top:10px; color:#2b2b2b;">
Before running the evaluation, it is useful to review the configuration settings currently stored inside the <code>Evaluation</code> object.
The next cell uses <code>ev.configurations.to_pandas()</code> to display these settings as a Pandas DataFrame.

In [ ]:
# Display the configuration settings as a Pandas DataFrame
ev.configurations.to_pandas()

<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; color:#2b2b2b; margin-top:10px;">
The configuration table now shows three registered datasets inside the <code>Evaluation</code> object.
Each entry defines a data source used during the evaluation process:
</p>
<ul style="font-family:'Segoe UI', sans-serif; line-height:1.6; color:#2b2b2b;">
  <li><strong>nwm30_retrospective</strong> — a <em>secondary</em> dataset containing the NWM 3.0 retrospective model outputs.</li>
  <li><strong>usgs_observations</strong> — the <em>primary</em> dataset, representing the USGS observed streamflow.</li>
  <li><strong>gage_10109001</strong> — an additional <em>secondary</em> dataset describing the NGIAB (NextGen) simulation output for the selected gage.</li>
</ul>

<h4 style="background-color:#e9f0ff; color:#1b2a4e; padding:10px 14px; border-left:5px solid #4b6cff;
           border-radius:6px; font-family:'Segoe UI', sans-serif; font-size:17px; margin-top:22px;">
  🔹 Load the NGIAB Streamflow Time Series into the Evaluation
</h4>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; margin-top:10px; color:#2b2b2b;">
In this step, we load the NGIAB simulated streamflow data from the subsetted
<code>troute</code> NetCDF file into the <code>Evaluation</code> object.  
This registers the NGIAB time series as a <strong>secondary dataset</strong>, allowing TEEHR to align it later with
the USGS observations for comparison.
</p>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.6; color:#2b2b2b; margin-top:10px;">
The <code>field_mapping</code> tells TEEHR how to interpret the NetCDF fields, while
<code>constant_field_values</code> provides metadata such as units, variable names,
and the configuration associated with this dataset.
Using <code>location_id_prefix="ngen"</code> ensures that location IDs in the time series
match the crosswalk entries created earlier.
</p>

In [ ]:
ev.secondary_timeseries.load_netcdf(
    in_path=troute_subset_filepath,
    field_mapping={
        "time": "value_time",
        "feature_id": "location_id",
        "flow": "value"
    },
    constant_field_values={
        "unit_name": "m^3/s",
        "variable_name": "streamflow_hourly_inst",
        "configuration_name": NGIAB_CONFIGURATION_NAME,
        "reference_time": None
    },
    location_id_prefix="ngen"
)

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:14px 18px; border-radius:8px; margin-top:0; margin-bottom:12px;">
  <h3 style="margin:0; font-size:21px; font-weight:700; color:#eaf7ff; font-family:'Segoe UI', sans-serif;">
    10. Visualize NGIAB, NWM, and USGS Streamflow Time Series
  </h3>
</div>
<p style="font-family:'Segoe UI', sans-serif; line-height:1.65; color:#2b2b2b; margin-top:8px;">
In this step, we extract and plot the streamflow time series from all three datasets used in the evaluation:
</p>
<ul style="font-family:'Segoe UI', sans-serif; line-height:1.6; color:#2b2b2b; margin-top:6px;">
  <li><strong>NGIAB (NextGen)</strong> simulation output for the selected routing segment</li>
  <li><strong>NWM 3.0 retrospective</strong> model output for the corresponding segment</li>
  <li><strong>USGS</strong> observed streamflow for the matching gage</li>
</ul>

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.transforms as mtransforms
import pandas as pd
import numpy as np
import spotpy
from matplotlib.gridspec import GridSpec

# ----------------------------------------------------
# 0. Helper functions
# ----------------------------------------------------
def kge_spotpy(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    mask = (~np.isnan(obs)) & (~np.isnan(sim))
    if mask.sum() == 0:
        return np.nan
    return spotpy.objectivefunctions.kge(obs[mask], sim[mask])

def nse_like_calibration(obs, sim):
    obs = np.asarray(obs, dtype=float)
    sim = np.asarray(sim, dtype=float)
    mask = (~np.isnan(obs)) & (~np.isnan(sim))
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.all(obs == obs[0]):
        return np.nan
    return 1.0 - np.sum((obs - sim) ** 2) / np.sum((obs - np.mean(obs)) ** 2)

def _fmt_metric(x):
    return "—" if (x is None or np.isnan(x)) else f"{x:.3f}"

# ----------------------------------------------------
# 1. Query time series from TEEHR
# ----------------------------------------------------
ngen_df = ev.secondary_timeseries.query(
    filters=["configuration_name = 'ngen_simulation'", "location_id = 'ngen-2861391'"]
).to_pandas()

nwm_df = ev.secondary_timeseries.query(
    filters=["configuration_name = 'nwm30_retrospective'", "location_id = 'nwm30-664424'"]
).to_pandas()

usgs_df = ev.primary_timeseries.query(
    filters=["configuration_name = 'usgs_observations'", "location_id = 'usgs-10109001'"]
).to_pandas()

# ----------------------------------------------------
# 2. Clean time columns + extent
# ----------------------------------------------------
for df in (ngen_df, nwm_df, usgs_df):
    df["value_time"] = pd.to_datetime(df["value_time"])
    df.sort_values("value_time", inplace=True)

start = min(ngen_df["value_time"].min(), nwm_df["value_time"].min(), usgs_df["value_time"].min())
end   = max(ngen_df["value_time"].max(), nwm_df["value_time"].max(), usgs_df["value_time"].max())

spinup_end  = pd.Timestamp("2019-09-30")
calib_start = pd.Timestamp("2019-10-01")
spinup_end  = max(min(spinup_end, end), start)
calib_start = max(min(calib_start, end), start)

# ----------------------------------------------------
# 3. Merge for metrics (USGS as reference)
# ----------------------------------------------------
merged = pd.DataFrame({"time": usgs_df["value_time"], "obs": usgs_df["value"]})

merged = merged.merge(
    ngen_df[["value_time", "value"]].rename(columns={"value_time": "time", "value": "sim_ngen"}),
    on="time", how="left"
)
merged = merged.merge(
    nwm_df[["value_time", "value"]].rename(columns={"value_time": "time", "value": "sim_nwm"}),
    on="time", how="left"
)

spin_df  = merged[(merged["time"] >= start) & (merged["time"] <= spinup_end)]
calib_df = merged[(merged["time"] >= calib_start) & (merged["time"] <= end)]

KGE_spin_ngen = kge_spotpy(spin_df["obs"].values,  spin_df["sim_ngen"].values)
NSE_spin_ngen = nse_like_calibration(spin_df["obs"].values, spin_df["sim_ngen"].values)

KGE_spin_nwm  = kge_spotpy(spin_df["obs"].values,  spin_df["sim_nwm"].values)
NSE_spin_nwm  = nse_like_calibration(spin_df["obs"].values, spin_df["sim_nwm"].values)

KGE_cal_ngen  = kge_spotpy(calib_df["obs"].values, calib_df["sim_ngen"].values)
NSE_cal_ngen  = nse_like_calibration(calib_df["obs"].values, calib_df["sim_ngen"].values)

KGE_cal_nwm   = kge_spotpy(calib_df["obs"].values, calib_df["sim_nwm"].values)
NSE_cal_nwm   = nse_like_calibration(calib_df["obs"].values, calib_df["sim_nwm"].values)

# ----------------------------------------------------
# 4. Publication style
# ----------------------------------------------------
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig = plt.figure(figsize=(12, 6))
gs = GridSpec(2, 1, height_ratios=[4, 0.75], hspace=0.06)

ax = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1], sharex=ax)

# ----------------------------------------------------
# 5. Plot series
# ----------------------------------------------------
ax.plot(usgs_df["value_time"], usgs_df["value"],
        label="Observed (USGS 10109001)",
        color="black", linewidth=2.0, zorder=3)

ax.plot(nwm_df["value_time"], nwm_df["value"],
        label="NWM v3.0 Retrospective",
        color="#ff7f0e", linestyle="--", linewidth=1.6, alpha=0.9, zorder=2)

ax.plot(ngen_df["value_time"], ngen_df["value"],
        label="NextGen Simulation",
        color="#1f77b4", linestyle="--", linewidth=1.6, alpha=0.9, zorder=2)

# Professional split marker line (behind data)
ax.axvline(
    spinup_end,
    color="0.45",
    linestyle=":",
    linewidth=0.9,
    zorder=0
)

ax.set_ylabel("Streamflow (m³/s)")
ax.grid(True, alpha=0.25, linestyle="--")

# ----------------------------------------------------
# 6. Reserve a header band (no overlap with hydrograph)
# ----------------------------------------------------
HEADER_BAND = 0.18
DATA_TOP = 1.0 - HEADER_BAND

ymin, ymax = ax.get_ylim()
ymin = min(0, ymin)
data_range = (ymax - ymin)
if data_range <= 0:
    data_range = 1.0

new_top = ymin + data_range / DATA_TOP
ax.set_ylim(ymin, new_top)

# Separator line between data region and header band
ax.axhline(
    ymin + (new_top - ymin) * DATA_TOP,
    color="0.85", linewidth=0.8, zorder=0
)

plt.setp(ax.get_xticklabels(), visible=False)

# ----------------------------------------------------
# 6b. Background shading for Spin-up and Calibration (MAIN AXIS)
#     (matches bottom bar colors; stays below header band)
# ----------------------------------------------------
ax.axvspan(
    start, spinup_end,
    ymin=0.0, ymax=DATA_TOP,
    facecolor="0.92",
    edgecolor="none",
    zorder=0
)
ax.axvspan(
    calib_start, end,
    ymin=0.0, ymax=DATA_TOP,
    facecolor="#e6f2ff",
    edgecolor="none",
    zorder=0
)

# ----------------------------------------------------
# 7. Legend (frameless) in header band
# ----------------------------------------------------
ax.legend(
    loc="upper left",
    frameon=False,
    bbox_to_anchor=(0.01, 0.995),
    borderaxespad=0.0,
    handlelength=2.4,
)

# ----------------------------------------------------
# 8. Metrics boxes (same y-level, symmetric around split, header band)
# ----------------------------------------------------
spin_text = (
    "Spin-up\n"
    f"NextGen   KGE {_fmt_metric(KGE_spin_ngen)}   NSE {_fmt_metric(NSE_spin_ngen)}\n"
    f"NWM 3.0   KGE {_fmt_metric(KGE_spin_nwm)}   NSE {_fmt_metric(NSE_spin_nwm)}"
)
calib_text = (
    "Calibration\n"
    f"NextGen   KGE {_fmt_metric(KGE_cal_ngen)}   NSE {_fmt_metric(NSE_cal_ngen)}\n"
    f"NWM 3.0   KGE {_fmt_metric(KGE_cal_nwm)}   NSE {_fmt_metric(NSE_cal_nwm)}"
)

box_kw = dict(
    boxstyle="round,pad=0.35",
    facecolor="white",
    edgecolor="0.6",
    linewidth=0.8,
    alpha=0.98,
)

# Symmetric horizontal placement around split
left_span  = spinup_end - start
right_span = end - spinup_end
delta = 0.63 * min(left_span, right_span)  # adjust spacing from split here
x_left  = spinup_end - delta
x_right = spinup_end + delta

blend = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)

# Vertical placement within header band (lower = smaller value)
y_level = 0.74

ax.text(
    x_left, y_level, spin_text,
    transform=blend,
    ha="center", va="top",
    fontsize=9,
    family="DejaVu Sans Mono",
    bbox=box_kw,
    zorder=10,
)

ax.text(
    x_right, y_level, calib_text,
    transform=blend,
    ha="center", va="top",
    fontsize=9,
    family="DejaVu Sans Mono",
    bbox=box_kw,
    zorder=10,
)

# ----------------------------------------------------
# 9. Bottom period bar
# ----------------------------------------------------
ax_bar.set_ylim(0, 1)
ax_bar.set_yticks([])

ax_bar.axvspan(start, spinup_end, facecolor="0.85", alpha=1.0)
ax_bar.axvspan(calib_start, end, facecolor="#d0e7ff", alpha=1.0)

spin_mid  = start + (spinup_end - start) / 2
calib_mid = calib_start + (end - calib_start) / 2

ax_bar.text(spin_mid, 0.5, "Model Spin-up\n(2017-10-01 – 2019-09-30)",
            ha="center", va="center", fontsize=10)
ax_bar.text(calib_mid, 0.5, "Model Calibration\n(2019-10-01 – 2021-09-30)",
            ha="center", va="center", fontsize=10)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["left"].set_visible(False)

ax_bar.set_xlabel("Date/Time")
ax_bar.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax_bar.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()

fig.subplots_adjust(top=0.96, left=0.08, right=0.98, bottom=0.12, hspace=0.06)

# ----------------------------------------------------
# 10. Save
# ----------------------------------------------------
output_path = "/home/jovyan/streamflow_spinup_calibration_metrics_headerband_shaded.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
print("Saved plot to:", output_path)

plt.show()
plt.close(fig)
